In [20]:
import time
import numpy as np

np.set_printoptions(precision=4, suppress=True, linewidth=150)

def khatri_rao(matrices):
    result = matrices[0]

    for matrix in matrices[1:]:
        # result: (I, R)
        # matrix: (J, R)
        result = (
            result[:, None, :] * matrix[None, :, :]
        ).reshape(-1, result.shape[1], order="F")

    return result


def unfold(X, mode):
    """
    Mode-n unfolding của tensor.
    """
    axes = [mode] + [i for i in range(X.ndim) if i != mode]

    return np.transpose(X, axes).reshape(
        X.shape[mode], -1, order="F"
    )


def cp_als(X, rank, max_iter=100, tol=1e-6, verbose=True):
    shape = X.shape
    N = X.ndim

    # Khởi tạo factor matrices
    factors = [
        np.random.rand(dim, rank)
        for dim in shape
    ]

    weights = np.ones(rank)

    norm_X = np.linalg.norm(X)

    if norm_X == 0:
        return weights, factors

    errors = []
    prev_err = np.inf

    start_time = time.time()

    if verbose:
        print(
            f"{'Lần lặp':<15} | "
            f"{'Sai số tương đối':<25} | "
            f"{'Delta Error (Error thay đổi từ vòng trước đến vòng hiện tại)':<20}"
        )
        print("-" * 70)

    unfolded = [unfold(X, n) for n in range(N)]

    for iteration in range(max_iter):

        for n in range(N):

            # Các factor matrix khác mode n
            other_factors = [
                factors[i]
                for i in range(N)
                if i != n
            ]

            # V = G1 * G2 * ... * GN
            # với phép nhân Hadamard giữa các Gram matrices
            V = np.ones((rank, rank))

            for factor in other_factors:
                V *= factor.T @ factor

            # Tích Khatri-Rao
            kr = khatri_rao(other_factors)

            # A_n = X_(n) @ KR @ V^{-1}
            rhs = unfolded[n] @ kr

            try:
                new_factor = np.linalg.solve(
                    V.T,
                    rhs.T
                ).T
            except np.linalg.LinAlgError:
                new_factor = rhs @ np.linalg.pinv(V)

            # Chuẩn hóa factor
            weights = np.linalg.norm(
                new_factor,
                axis=0
            )

            safe_weights = np.where(
                weights == 0,
                1.0,
                weights
            )

            factors[n] = (
                new_factor / safe_weights
            )

        kr_all = khatri_rao(factors[1:])

        # X ≈ A0 * diag(weights) * KR^T
        X1_rec = (
            factors[0] * weights
        ) @ kr_all.T

        X_rec = X1_rec.reshape(
            shape,
            order="F"
        )

        err = np.linalg.norm(
            X - X_rec
        ) / norm_X

        errors.append(err)

        delta_err = (
            abs(prev_err - err)
            if np.isfinite(prev_err)
            else 0.0
        )

        if verbose:
            print(
                f"Lặp {iteration + 1:3d}/{max_iter:<3d} | "
                f"{err:<25.6f} | "
                f"{delta_err:<20.6e}"
            )

        # Kiểm tra hội tụ
        if (
            iteration > 0
            and delta_err < tol
        ):
            if verbose:
                print(
                    f" Hội tụ tại vòng lặp "
                    f"thứ {iteration + 1}"
                )
            break

        prev_err = err

    elapsed_time = time.time() - start_time

    if verbose:
        total_iters = len(errors)

        initial_err = (
            errors[0]
            if errors
            else 0
        )

        final_err = (
            errors[-1]
            if errors
            else 0
        )

        avg_speed = (
            (initial_err - final_err)
            / total_iters
            if total_iters > 0
            else 0
        )

        avg_time = (
            elapsed_time / total_iters
            if total_iters > 0
            else 0
        )

        print("-" * 70)
        print(
            f"  Thời gian tính toán       : "
            f"{elapsed_time:.4f} giây"
        )
        print(
            f"  Số iterations             : "
            f"{total_iters}"
        )
        print(
            f"  Thời gian / iteration     : "
            f"{avg_time:.6f} giây"
        )
        print(
            f"  Average error reduction   : "
            f"{avg_speed:.6e}"
        )

    return weights, factors


shape_str = input(
    "Kích thước tensor "
    "(ngăn cách bởi dấu phẩy,): "
)

shape = tuple(
    map(int, shape_str.split(","))
)

rank = int(
    input(
        "Nhập Rank R "
        "(số lượng component): "
    )
)

np.random.seed(42)

X = np.random.rand(*shape)

print("\n" + "=" * 70)
print(
    f" TENSOR {X.ndim}-CHIỀU "
    f"KÍCH THƯỚC: {shape}"
)
print("=" * 70)

if X.ndim >= 3:

    print("\n Frontal Slices")

    for k in range(shape[2]):

        slice_k = X[:, :, k, ...]

        print(
            f"\nFrontal Slice k = {k + 1} "
            f"(shape = {slice_k.shape})"
        )

        print(slice_k)

else:

    print(
        "\nTensor có số chiều < 3:"
    )
    print(X)

print("\n" + "=" * 70)
print(" UNFOLDED MATRICES")
print("=" * 70)

for n in range(X.ndim):

    X_n = unfold(X, n)

    print(
        f"\nMode-{n + 1} unfolding "
        f"(shape = {X_n.shape}):"
    )

    print(X_n)


print("\n" + "=" * 70)
print(
    f" CP-ALS (RANK R = {rank})"
)
print("=" * 70)

weights, factors = cp_als(
    X,
    rank=rank,
    max_iter=100,
    tol=1e-6,
    verbose=True
)


print("=" * 70)
print("\n MA TRẬN NHÂN TỐ KẾT QUẢ")

print(
    f"\nVector trọng số Lambda "
    f"(kích thước {len(weights)}):"
)

print(weights)

print("\n" + "-" * 50)

for mode, factor in enumerate(factors, start=1):

    rows, cols = factor.shape

    print(
        f"\nMa trận Mode-{mode} "
        f"({rows} x {cols}):"
    )

    headers = "      " + "  ".join(
        f"Comp {r + 1}"
        for r in range(cols)
    )

    print(headers)

    for row_idx, row in enumerate(factor, start=1):

        values = "  ".join(
            f"{value:10.4f}"
            for value in row
        )

        print(
            f"Hàng {row_idx:2d}: {values}"
        )

    print("-" * 50)


Kích thước tensor (ngăn cách bởi dấu phẩy,):  5,5,5
Nhập Rank R (số lượng component):  3



 TENSOR 3-CHIỀU KÍCH THƯỚC: (5, 5, 5)

 Frontal Slices

Frontal Slice k = 1 (shape = (5, 5))
[[0.3745 0.156  0.0206 0.1834 0.6119]
 [0.7852 0.6075 0.8084 0.122  0.6625]
 [0.9696 0.9219 0.3887 0.5427 0.7722]
 [0.729  0.8631 0.3252 0.1196 0.4938]
 [0.0314 0.2493 0.2898 0.8715 0.8074]]

Frontal Slice k = 2 (shape = (5, 5))
[[0.9507 0.0581 0.9699 0.3042 0.1395]
 [0.1997 0.1705 0.3046 0.4952 0.3117]
 [0.7751 0.0885 0.2713 0.1409 0.1987]
 [0.7713 0.6233 0.7296 0.7132 0.5227]
 [0.6364 0.4104 0.1612 0.8037 0.8961]]

Frontal Slice k = 3 (shape = (5, 5))
[[0.732  0.8662 0.8324 0.5248 0.2921]
 [0.5142 0.0651 0.0977 0.0344 0.5201]
 [0.9395 0.196  0.8287 0.8022 0.0055]
 [0.074  0.3309 0.6376 0.7608 0.4275]
 [0.3144 0.7556 0.9297 0.1866 0.318 ]]

Frontal Slice k = 4 (shape = (5, 5))
[[0.5987 0.6011 0.2123 0.4319 0.3664]
 [0.5924 0.9489 0.6842 0.9093 0.5467]
 [0.8948 0.0452 0.3568 0.0746 0.8155]
 [0.3585 0.0636 0.8872 0.5613 0.0254]
 [0.5086 0.2288 0.8081 0.8926 0.1101]]

Frontal Slice k = 5 (shape 